In [1]:
library(fixest)
library(modelsummary)
library(gt)
library(marginaleffects)
library(ggeffects)
library(ggplot2)
library(dplyr)
library(sjPlot)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Learn more about sjPlot with 'browseVignettes("sjPlot")'.



In [2]:
data = read.csv('plos_regression.csv')
head(data)

,ost_bk,year,dis,alpha,cluster_id,seqno,cntry,credit,role,Code,class,intl,gender,yfp,npub
,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<dbl>,<dbl>
1,65855275,2018,64,noalp,83846425,4,ESP,Investigation,M,ESP,Proficient,3,M,2007,4.317488
2,65855275,2018,64,noalp,83846425,4,ESP,Writing - Original Draft,M,ESP,Proficient,3,M,2007,4.317488
3,65855275,2018,64,noalp,83846425,4,ESP,Writing - Review & Editing,M,ESP,Proficient,3,M,2007,4.317488
4,66890753,2018,18,noalp,19277330,8,ESP,Supervision,L,ESP,Proficient,2,U,1987,6.361302
5,66890753,2018,18,noalp,73838998,7,ESP,Formal analysis,M,ESP,Proficient,2,M,2011,2.995732
6,66890753,2018,18,noalp,73838998,7,ESP,Validation,M,ESP,Proficient,2,M,2011,2.995732


In [3]:
credits = unique(data$credit)
print(credits)

 [1] "Investigation"              "Writing - Original Draft"  
 [3] "Writing - Review & Editing" "Supervision"               
 [5] "Formal analysis"            "Validation"                
 [7] "Methodology"                "Funding acquisition"       
 [9] "Project administration"     "Resources"                 
[11] "Data curation"              "Conceptualization"         
[13] "Visualization"              "Software"                  


In [4]:
model_results = list()
for (c in credits){
    df = data
    df$dep_var = ifelse(df$credit==c, 1, 0)
    df = df %>% select('ost_bk','cluster_id','class','gender','yfp','npub','dep_var')
    df = df %>% distinct()
    df = df %>% 
    group_by(cluster_id) %>% 
    filter(!(dep_var == 0 & any(dep_var == 1))) %>% 
    ungroup()
    model = feglm(dep_var~gender+npub+class|ost_bk, data=df, family=binomial())
    model_results[[c]]=model
}

NOTE: 917 fixed-effects (5,600 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 413 fixed-effects (2,170 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 1,625 fixed-effects (11,692 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 694 fixed-effects (4,315 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 564 fixed-effects (3,288 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 1,967 fixed-effects (14,105 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 893 fixed-effects (5,488 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 1,100 fixed-effects (7,317 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 1,239 fixed-effects (8,183 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 1,756 fixed-effects (11,969 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 1,031 fixed-effects (7,014 observations) 

In [5]:
table = modelsummary(model_results[1:7], output='gt', stars=TRUE,
            coef_map=c('genderM'='Male','genderU'='Gender(Unknown)','npub'='Log no.pub',
                          'classProficient'='Proficient','classDeveloping'='Developing','classLagging'='Lagging'),
            gof_map = c("nobs", "r.squared","adj.r.squared")) |> gtsave(file='credit_regression_1.html')
table = modelsummary(tail(model_results,7), output='gt', stars=TRUE,
            coef_map=c('genderM'='Male','genderU'='Gender(Unknown)','npub'='Log no.pub',
                          'classProficient'='Proficient','classDeveloping'='Developing','classLagging'='Lagging'),
            gof_map = c("nobs", "r.squared","adj.r.squared")) |> gtsave(file='credit_regression_2.html')

In [6]:
credits

[1] "Investigation"              "Writing - Original Draft"  
 [3] "Writing - Review & Editing" "Supervision"               
 [5] "Formal analysis"            "Validation"                
 [7] "Methodology"                "Funding acquisition"       
 [9] "Project administration"     "Resources"                 
[11] "Data curation"              "Conceptualization"         
[13] "Visualization"              "Software"

In [7]:
# Define leading roles and authorship types
leading_roles <- c('Methodology', 'Conceptualization', 'Supervision')
authorship_roles <- c('F', 'M', 'L')

# Start with the initial data and create a new variable 'dep_var'
df <- data
df$dep_var <- ifelse(df$credit %in% leading_roles, 1, 0)

# Remove rows where dep_var is 0 only if the id has both 1 and 0 for the same group (ost_bk and cluster_id)
df <- df %>%
  group_by(ost_bk, cluster_id) %>%
  filter(!(dep_var == 0 & any(dep_var == 1))) %>%
  ungroup()

# Function to filter out rows where all dep_var values are either 1 or 0 across groups
remove_invalid_ost_bk <- function(df, filter_data) {
  result <- df %>%
    group_by(ost_bk) %>%
    filter(all(dep_var == 1) | all(dep_var == 0)) %>%
    distinct(ost_bk)
  
  invalid_list <- as.list(result$ost_bk)
  filter_data %>% filter(!ost_bk %in% invalid_list)
}

# Apply the function to filter based on leading roles
data_filtered <- remove_invalid_ost_bk(df, data)

# Loop through each authorship role and apply the function
for (role in authorship_roles) {
  # Create dep_var based on the 'role' column
  data_filtered$dep_var <- ifelse(data_filtered$role == role, 1, 0)
  
  # Apply the function to filter out additional invalid ost_bk values
  data_filtered <- remove_invalid_ost_bk(data_filtered, data_filtered)
}

In [14]:
model_results = list()
leading = c('Methodology','Conceptualization','Supervision')
df = data_filtered
df$dep_var = ifelse(df$credit %in% leading, 1, 0)
df = df %>% select('ost_bk','cluster_id','class','gender','yfp','npub','dep_var')
df = df %>% distinct()
# Remove rows with value 0 only if the id has both 1 and 0
df <- df %>%
  group_by(ost_bk,cluster_id) %>%
  filter(!(dep_var == 0 & any(dep_var == 1))) %>%
  ungroup()
model = feglm(dep_var~gender+npub+class|ost_bk, data=df, family=binomial())
model_results[['Strategic Role']]=model

In [15]:
df = subset(data_filtered, alpha=='noalp')
df$leading = ifelse(df$credit %in% leading, 1, 0)
df = df %>% select('ost_bk','cluster_id','class','gender','yfp','npub','leading','role')
df = df %>% distinct()
df = df %>%
  group_by(ost_bk,cluster_id) %>%
  filter(!(leading == 0 & any(leading == 1))) %>%
  ungroup()

In [16]:
authorship = c('F','M','L')
authorship_label = c('First Author','Middle Author','Last Author')
for (i in seq_along(authorship)){
    role = authorship[i]
    label = authorship_label[i]
    
    df$dep_var = ifelse(df$role==role, 1, 0)
    model = feglm(dep_var~gender+npub+class+leading|ost_bk, data=df, family=binomial())
    model_results[[label]]=model
}

In [17]:
table = modelsummary(model_results, output='gt', stars=TRUE,
            coef_map=c('genderM'='Male','genderU'='Gender(Unknown)','npub'='Log no.pub',
                          'classProficient'='Proficient','classDeveloping'='Developing','classLagging'='Lagging',
                      'leading'='Strategic Role'),
                     
            gof_map = c("nobs", "r.squared","adj.r.squared")) |> gtsave(file='credit_regression_leading.html')

In [8]:
model_results

$`Intellectual Role`
GLM estimation, family = binomial, Dep. Var.: dep_var
Observations: 25,379 
Fixed-effects: ost_bk: 2,882
Standard-errors: Clustered (ost_bk) 
                 Estimate Std. Error   t value   Pr(>|t|)    
genderM          0.051146   0.039506  1.294655 1.9544e-01    
genderU         -0.051445   0.056124 -0.916641 3.5933e-01    
npub             0.324537   0.012562 25.834790  < 2.2e-16 ***
classDeveloping -0.458126   0.129918 -3.526274 4.2145e-04 ***
classLagging    -0.606003   0.077766 -7.792675 6.5605e-15 ***
classProficient  0.048879   0.061384  0.796293 4.2586e-01    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -13,757.6   Adj. Pseudo R2: 0.035792
           BIC:  56,804.4     Squared Cor.: 0.254716

$`First Author`
GLM estimation, family = binomial, Dep. Var.: dep_var
Observations: 25,278 
Fixed-effects: ost_bk: 2,854
Standard-errors: Clustered (ost_bk) 
                 Estimate Std. Error   t value   Pr(>|t|)    
genderM  